
# V48 LIVE — tb-BanglaBERT Matched R-Drop Experiment

**Protected public-LB champion:** V38 = **0.608**  
**New minimum public-LB goal:** **0.609**  
**Main target:** **0.620**

This notebook implements the single controlled experiment chosen after the V42–V47 failures and the Astra research audit.

## Hypothesis

> On a transliteration-aware Bangla encoder, adding a fixed Bernoulli R-Drop consistency penalty improves fresh group-heldout toxicity generalization over an otherwise compute-matched control.

## Why this checkpoint?

`aplycaebous/tb-BanglaBERT-fpt`

- BanglaBERT/ELECTRA family further pretrained on Romanized/transliterated Bangla.
- Hugging Face currently displays **MIT** for this repository.
- The exact repository revision is pinned below.
- The project audit found no prior executed experiment on this exact checkpoint.

## Controlled comparison

Each outer fold trains TWO otherwise matched models:

**CONTROL**
- two stochastic dropout forward passes
- weighted multilabel BCE averaged across the two passes

**R-DROP**
- the exact same two-pass BCE
- plus symmetric **Bernoulli** KL consistency loss
- fixed `alpha = 1.0`

Nothing else is tuned between the two arms.

## Evaluation design

- primary evaluation uses only the 3297 LIVE training rows
- 3 fresh outer group-heldout folds
- normalized duplicate/conflicting texts never cross folds
- 20% of each outer-training group set is held out for calibration
- thresholds are chosen only on inner calibration groups
- outer labels are scored exactly once
- historical 500-row validation is only a secondary diagnostic
- test probabilities are saved for a later integration step, but this notebook does **not** generate a Kaggle challenger submission

## Pre-registered promotion gate

V48 passes only if ALL are true:

- mean toxicity-only official-score gain >= **0.004**
- mean toxicity macro-F1 gain >= **0.008**
- positive primary gain in at least **2 of 3 folds**
- no fold primary gain below **-0.005**
- mean toxicity AP difference >= 0 and no fold AP difference below -0.01
- pooled toxicity macro-F1 degradation no worse than -0.015 in either script subgroup
- full-model pooled implicit and threatening F1 do not each fall by more than 0.02
- zero group leakage
- total run time <= 300 minutes

The **0.609 leaderboard goal does not weaken this offline gate**. The stronger gate is intentional because small local gains have repeatedly failed on the public leaderboard.

At the end, upload `v48_results.zip` to ChatGPT. Do not submit anything to Kaggle from this notebook.


In [ ]:

# 1. Upload V48 LIVE bundle
from google.colab import files
import pathlib, zipfile

uploaded = files.upload()
assert "v48_LIVE_rdrop_matched_data.zip" in uploaded

DATA_DIR = pathlib.Path("/content/v48_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v48_LIVE_rdrop_matched_data.zip") as z:
    z.extractall(DATA_DIR)

for p in sorted(DATA_DIR.iterdir()):
    print(p.name, f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Install pinned runtime dependencies

import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn>=1.4",
    "scipy",
    "safetensors",
    "huggingface_hub",
    "tqdm",
])

print("✅ dependencies installed")


In [ ]:

# 3. Imports, configuration, exact model revision

import gc, re, os, time, math, json, random, hashlib, unicodedata, platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

import sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    average_precision_score,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from huggingface_hub import snapshot_download

SEED = 2026

LABELS = [
    "is_toxic",
    "implicit_toxicity",
    "sarcasm",
    "contains_profanity",
    "threatening",
    "identity_attack",
    "sexual_content",
]
LI = {l:i for i,l in enumerate(LABELS)}

MODEL_REPO = "aplycaebous/tb-BanglaBERT-fpt"
MODEL_REVISION = "0b24caaab62d12457ed0eb00ff0f459a1ea509bc"

N_OUTER = 3
CALIB_FRACTION = 0.20

MAX_LENGTH = 128
TRAIN_BATCH = 8
EVAL_BATCH = 48
ACCUM = 2
EPOCHS = 3
SNAPSHOT_EPOCHS = [2, 3]

BACKBONE_LR = 1.2e-5
HEAD_LR = 2.5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
RDROP_ALPHA = 1.0

THRESH_GRID = np.round(np.arange(0.05, 0.951, 0.05), 2)

PROTECTED_PUBLIC_LB = 0.608
MIN_PUBLIC_LB_GOAL = 0.609
MAIN_PUBLIC_LB_GOAL = 0.620

USE_DRIVE_CACHE = True

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = Path("/content/drive/MyDrive/V48_RDrop_tbBanglaBERT")
else:
    RUN_DIR = Path("/content/v48_run")

RUN_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = Path("/content/v48_export")
EXPORT_DIR.mkdir(exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(SEED)

assert torch.cuda.is_available(), "Enable a GPU runtime (T4 preferred)."

print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__)
print("Transformers runtime will be recorded later.")
print("Run dir:", RUN_DIR)


In [ ]:

# 4. Download exact model revision once, record hashes and license metadata

MODEL_DIR = Path("/content/tb_banglabert_fpt_pinned")

if not MODEL_DIR.exists() or not any(MODEL_DIR.iterdir()):
    local = snapshot_download(
        repo_id=MODEL_REPO,
        revision=MODEL_REVISION,
        local_dir=str(MODEL_DIR),
    )
    print("Downloaded to:", local)
else:
    print("Reusing local model directory.")

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

model_hashes = {}
for name in [
    "config.json",
    "vocab.txt",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "pytorch_model.bin",
]:
    p = MODEL_DIR / name
    if p.exists():
        model_hashes[name] = {
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        }

print(json.dumps(model_hashes, indent=2)[:4000])

# Hugging Face currently displays MIT for this repository.
MODEL_LICENSE_NOTE = "Hugging Face repository page displays License: mit"


In [ ]:

# 5. Load and verify exact LIVE partition + data hashes

train = pd.read_csv(DATA_DIR/"train.csv", dtype={"ID":str})
valid500 = pd.read_csv(DATA_DIR/"validation.csv", dtype={"ID":str})
test = pd.read_csv(DATA_DIR/"test.csv", dtype={"ID":str})
champ = pd.read_csv(DATA_DIR/"submission_v38_champion.csv", dtype={"ID":str})

with open(DATA_DIR/"v48_manifest.json") as f:
    supplied_manifest = json.load(f)

assert len(train) == 3297
assert len(valid500) == 500
assert len(test) == 1000
assert test.ID.tolist() == champ.ID.tolist()
assert test.ID.head(4).tolist() == [
    "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
]
assert champ[LABELS].isin([0,1]).all().all()

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda:f.read(1024*1024), b""):
            h.update(b)
    return h.hexdigest()

for name in ["train.csv","validation.csv","test.csv","submission_v38_champion.csv"]:
    got = file_sha256(DATA_DIR/name)
    expected = supplied_manifest["files"][name]["sha256"]
    assert got == expected, f"Hash mismatch: {name}"

print("✅ exact LIVE 3297/500/1000 verified")
print("✅ accepted test IDs verified")
print("✅ protected V38 verified")
print("✅ bundle hashes verified")
print("V38 positive counts:", champ[LABELS].sum().to_dict())


In [ ]:

# 6. Deterministic grouping and script labels
#
# Group normalization is only for leakage prevention.
# Model input preserves punctuation/emoji and does NOT use the external
# CSEBUET normalizer package, avoiding a separate CC-BY-NC-SA code dependency.

BN_RE = re.compile(r"[\u0980-\u09FF]")
LAT_RE = re.compile(r"[A-Za-z]")

def model_text(x):
    x = unicodedata.normalize("NFKC", str(x))
    x = x.replace("\u200c", " ").replace("\u200d", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def group_key(x):
    # Latin lower-casing + deterministic Unicode/whitespace normalization.
    return model_text(x).lower()

def script_type(x):
    s = str(x)
    bn = bool(BN_RE.search(s))
    lat = bool(LAT_RE.search(s))
    if bn and lat:
        return "mixed"
    if bn:
        return "native"
    return "no_bengali"

for df in [train, valid500, test]:
    df["model_text"] = df["text"].map(model_text)
    df["group_key"] = df["text"].map(group_key)
    df["script"] = df["model_text"].map(script_type)

print("Unique TRAIN groups:", train.group_key.nunique())
print("Duplicate extra rows:", len(train)-train.group_key.nunique())
print("TRAIN script counts:")
display(train.script.value_counts())


In [ ]:

# 7. Freeze three OUTER group folds, then one 20% CALIBRATION group split
#    inside each outer-training portion.
#
# Splitting happens at GROUP level, so duplicate/conflicting texts cannot cross.

group_rows = []

for g, d in train.groupby("group_key", sort=True):
    rec = {
        "group_key": g,
        "rows": len(d),
        "script": d["script"].mode().iloc[0],
    }
    for l in LABELS:
        rec[l] = int(d[l].max())
    group_rows.append(rec)

groups = pd.DataFrame(group_rows)

def make_group_strata(gdf, min_count):
    rare_any = (
        (gdf["implicit_toxicity"]==1)
        | (gdf["threatening"]==1)
        | (gdf["sexual_content"]==1)
    ).astype(int)

    mid_any = (
        (gdf["sarcasm"]==1)
        | (gdf["contains_profanity"]==1)
        | (gdf["identity_attack"]==1)
    ).astype(int)

    s = (
        gdf["is_toxic"].astype(str)
        + "_" + rare_any.astype(str)
        + "_" + mid_any.astype(str)
        + "_" + gdf["script"].astype(str)
    )

    counts = s.value_counts()
    s2 = s.copy()

    rare = set(counts[counts < min_count].index)
    if rare:
        simple = (
            gdf["is_toxic"].astype(str)
            + "_" + rare_any.astype(str)
            + "_" + gdf["script"].astype(str)
        )
        s2 = pd.Series(
            [simple.iloc[i] if s.iloc[i] in rare else s.iloc[i] for i in range(len(s))],
            index=gdf.index
        )

    counts2 = s2.value_counts()
    rare2 = set(counts2[counts2 < min_count].index)
    if rare2:
        simpler = (
            gdf["is_toxic"].astype(str)
            + "_" + gdf["script"].astype(str)
        )
        s2 = pd.Series(
            [simpler.iloc[i] if s2.iloc[i] in rare2 else s2.iloc[i] for i in range(len(s2))],
            index=gdf.index
        )

    return s2.astype(str).to_numpy()

outer_strata = make_group_strata(groups, N_OUTER)

outer_skf = StratifiedKFold(
    n_splits=N_OUTER,
    shuffle=True,
    random_state=SEED
)

groups["outer_fold"] = -1

for fold, (_, gi) in enumerate(
    outer_skf.split(groups, outer_strata)
):
    groups.loc[groups.index[gi], "outer_fold"] = fold

assert (groups.outer_fold >= 0).all()

group_to_outer = dict(zip(groups.group_key, groups.outer_fold))
train["outer_fold"] = train.group_key.map(group_to_outer).astype(int)

split_records = []
inner_maps = {}

for fold in range(N_OUTER):
    outer_train_groups = groups[groups.outer_fold != fold].copy().reset_index(drop=True)

    inner_strata = make_group_strata(outer_train_groups, 5)

    inner_skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED + 100 + fold
    )

    # Pre-registered: use inner split #0 as the 20% calibration groups.
    inner_train_idx, calib_idx = next(
        inner_skf.split(outer_train_groups, inner_strata)
    )

    train_groups = set(outer_train_groups.iloc[inner_train_idx].group_key)
    calib_groups = set(outer_train_groups.iloc[calib_idx].group_key)
    outer_groups = set(groups.loc[groups.outer_fold==fold, "group_key"])

    assert train_groups.isdisjoint(calib_groups)
    assert train_groups.isdisjoint(outer_groups)
    assert calib_groups.isdisjoint(outer_groups)

    inner_maps[fold] = {
        "train_groups": train_groups,
        "calib_groups": calib_groups,
        "outer_groups": outer_groups,
    }

    for role, gs in [
        ("train", train_groups),
        ("calib", calib_groups),
        ("outer", outer_groups),
    ]:
        d = train[train.group_key.isin(gs)]
        rec = {
            "fold": fold,
            "role": role,
            "rows": len(d),
            "groups": len(gs),
            "native": int((d.script=="native").sum()),
            "mixed": int((d.script=="mixed").sum()),
            "no_bengali": int((d.script=="no_bengali").sum()),
        }
        for l in LABELS:
            rec[l+"_pos"] = int(d[l].sum())
        split_records.append(rec)

split_report = pd.DataFrame(split_records)
display(split_report)

# Leak check
for fold in range(N_OUTER):
    m = inner_maps[fold]
    assert len(m["train_groups"] & m["calib_groups"]) == 0
    assert len(m["train_groups"] & m["outer_groups"]) == 0
    assert len(m["calib_groups"] & m["outer_groups"]) == 0

split_report.to_csv(EXPORT_DIR/"v48_split_report.csv", index=False)
print("✅ zero group leakage by construction")


In [ ]:

# 8. Dataset / collator / metric helpers

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=True)

class MultiLabelDS(Dataset):
    def __init__(self, df, with_labels=True):
        self.text = df["model_text"].astype(str).tolist()
        self.y = df[LABELS].to_numpy(np.float32) if with_labels else None

    def __len__(self):
        return len(self.text)

    def __getitem__(self, i):
        item = tokenizer(
            self.text[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        if self.y is not None:
            item["labels"] = self.y[i]
        return item

class Collator:
    def __call__(self, items):
        y = None
        if "labels" in items[0]:
            y = torch.tensor(
                np.stack([x.pop("labels") for x in items]),
                dtype=torch.float32
            )
        batch = tokenizer.pad(
            items,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )
        if y is not None:
            batch["labels"] = y
        return batch

collator = Collator()

TASK_WEIGHT = torch.tensor(
    [0.50] + [0.50/6.0]*6,
    dtype=torch.float32
)

POS_BOOST = torch.tensor(
    [1.0, 1.5, 1.2, 1.2, 1.6, 1.1, 1.5],
    dtype=torch.float32
)

def binary_pos_f1(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def toxic_macro_f1(y, p):
    return f1_score(y, p, average="macro", labels=[0,1], zero_division=0)

def official_score(y_true, pred):
    tox = toxic_macro_f1(y_true[:,0], pred[:,0])
    attrs = [
        binary_pos_f1(y_true[:,j], pred[:,j])
        for j in range(1,7)
    ]
    return 0.5*tox + 0.5*np.mean(attrs)

def choose_threshold(y, prob, toxic=False):
    vals = []
    for t in THRESH_GRID:
        pred = (prob >= t).astype(int)
        s = toxic_macro_f1(y, pred) if toxic else binary_pos_f1(y, pred)
        vals.append((s, -abs(t-0.5), float(t)))
    vals.sort(reverse=True)
    return vals[0][2], vals[0][0]

def calibrate_thresholds(y, prob):
    th = []
    for j in range(7):
        t, _ = choose_threshold(
            y[:,j],
            prob[:,j],
            toxic=(j==0)
        )
        th.append(t)
    return np.array(th, float)

def apply_thresholds(prob, th):
    return (prob >= th[None,:]).astype(int)

def per_label_metrics(y, pred, prob):
    rows=[]
    for j,l in enumerate(LABELS):
        if j==0:
            f1 = toxic_macro_f1(y[:,j], pred[:,j])
        else:
            f1 = binary_pos_f1(y[:,j], pred[:,j])
        rows.append({
            "label": l,
            "precision_pos": precision_score(y[:,j], pred[:,j], zero_division=0),
            "recall_pos": recall_score(y[:,j], pred[:,j], zero_division=0),
            "f1_metric": f1,
            "ap": average_precision_score(y[:,j], prob[:,j]),
            "true_pos_support": int(y[:,j].sum()),
            "pred_pos_count": int(pred[:,j].sum()),
        })
    return pd.DataFrame(rows)


In [ ]:

# 9. Matched loss: two-pass BCE control vs BCE + Bernoulli R-Drop

def weighted_multilabel_bce(logits, y):
    raw = torch.nn.functional.binary_cross_entropy_with_logits(
        logits.float(),
        y,
        reduction="none"
    )

    task = TASK_WEIGHT.to(logits.device).unsqueeze(0)
    posb = POS_BOOST.to(logits.device).unsqueeze(0)

    class_w = torch.where(
        y > 0.5,
        posb.expand_as(y),
        torch.ones_like(y)
    )

    return (raw * class_w * task).sum(dim=1).mean()

def symmetric_bernoulli_kl(logits1, logits2):
    eps = 1e-6

    p = torch.sigmoid(logits1.float()).clamp(eps, 1-eps)
    q = torch.sigmoid(logits2.float()).clamp(eps, 1-eps)

    kl_pq = (
        p*(torch.log(p)-torch.log(q))
        +(1-p)*(torch.log(1-p)-torch.log(1-q))
    )

    kl_qp = (
        q*(torch.log(q)-torch.log(p))
        +(1-q)*(torch.log(1-q)-torch.log(1-p))
    )

    sym = 0.5*(kl_pq + kl_qp)

    task = TASK_WEIGHT.to(logits1.device).unsqueeze(0)
    return (sym * task).sum(dim=1).mean()

def sample_weights(df):
    script_w = df.script.map({
        "native":0.65,
        "mixed":1.60,
        "no_bengali":2.20,
    }).to_numpy(float)

    label_w = (
        1.0
        +1.5*df.implicit_toxicity.to_numpy(float)
        +0.5*df.sarcasm.to_numpy(float)
        +1.0*df.threatening.to_numpy(float)
        +1.0*df.sexual_content.to_numpy(float)
    )

    return script_w*label_w

def build_model(seed):
    seed_all(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        str(MODEL_DIR),
        num_labels=7,
        problem_type="multi_label_classification",
        ignore_mismatched_sizes=True,
    )
    return model

def build_optimizer(model):
    head=[]
    backbone=[]

    for n,p in model.named_parameters():
        if "classifier" in n or "score" in n:
            head.append(p)
        else:
            backbone.append(p)

    return torch.optim.AdamW([
        {"params":backbone, "lr":BACKBONE_LR, "weight_decay":WEIGHT_DECAY},
        {"params":head, "lr":HEAD_LR, "weight_decay":WEIGHT_DECAY},
    ])

@torch.no_grad()
def predict(model, loader):
    model.eval()
    parts=[]

    for batch in loader:
        batch.pop("labels", None)
        inp={k:v.to("cuda",non_blocking=True) for k,v in batch.items()}

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits=model(**inp).logits

        parts.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )

    return np.concatenate(parts)


In [ ]:

# 10. Optional throughput pilot BEFORE looking at any outer outcomes
#
# This is a compute check only. It does not inspect validation performance.

PILOT_BATCHES = 12

fold0 = inner_maps[0]
pilot_df = train[train.group_key.isin(fold0["train_groups"])].reset_index(drop=True)

pilot_model = build_model(SEED+999).to("cuda")
pilot_opt = build_optimizer(pilot_model)
pilot_scaler = torch.amp.GradScaler("cuda")

pilot_sampler = WeightedRandomSampler(
    torch.tensor(sample_weights(pilot_df), dtype=torch.double),
    num_samples=len(pilot_df),
    replacement=True,
    generator=torch.Generator().manual_seed(SEED+999),
)

pilot_loader = DataLoader(
    MultiLabelDS(pilot_df, True),
    batch_size=TRAIN_BATCH,
    sampler=pilot_sampler,
    num_workers=0,
    pin_memory=True,
    collate_fn=collator,
)

torch.cuda.synchronize()
t0=time.time()

pilot_model.train()
pilot_opt.zero_grad(set_to_none=True)

seen=0
for step,batch in enumerate(pilot_loader):
    if step>=PILOT_BATCHES:
        break

    y=batch.pop("labels").to("cuda")
    inp={k:v.to("cuda",non_blocking=True) for k,v in batch.items()}

    with torch.autocast(device_type="cuda",dtype=torch.float16):
        l1=pilot_model(**inp).logits
        l2=pilot_model(**inp).logits
        loss=0.5*(
            weighted_multilabel_bce(l1,y)
            +weighted_multilabel_bce(l2,y)
        )

    pilot_scaler.scale(loss).backward()
    pilot_opt.zero_grad(set_to_none=True)
    seen += 1

torch.cuda.synchronize()
elapsed=time.time()-t0
sec_per_batch=elapsed/max(seen,1)

# Estimate training microbatches across all folds/arms/epochs.
total_train_batches=0
for fold in range(N_OUTER):
    d=train[train.group_key.isin(inner_maps[fold]["train_groups"])]
    total_train_batches += math.ceil(len(d)/TRAIN_BATCH)*EPOCHS*2  # 2 arms

estimated_training_min = total_train_batches*sec_per_batch/60.0
estimated_total_min = estimated_training_min*1.20 + 25.0

print(f"Pilot: {sec_per_batch:.3f} sec/microbatch")
print(f"Estimated training time: {estimated_training_min:.1f} min")
print(f"Estimated total with 20% contingency + diagnostics: {estimated_total_min:.1f} min")

del pilot_model,pilot_opt,pilot_scaler,pilot_loader
gc.collect()
torch.cuda.empty_cache()

if estimated_total_min > 300:
    raise RuntimeError(
        "Projected V48 runtime exceeds the pre-registered 300-minute budget. "
        "Stop here and send this message to ChatGPT; do not silently reduce folds."
    )

print("✅ projected runtime fits the 5-hour budget")


In [ ]:

# 11. Train one arm for one outer fold
#
# Resume behavior: every completed fold/arm is cached to Drive as NPZ.
# If Colab disconnects, rerun the notebook; completed arms are skipped.

def make_eval_loader(df, with_labels):
    return DataLoader(
        MultiLabelDS(df.reset_index(drop=True), with_labels),
        batch_size=EVAL_BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collator,
    )

def train_arm(fold, arm):
    assert arm in ["control","rdrop"]

    cache = RUN_DIR/f"fold{fold}_{arm}_complete.npz"
    hist_path = RUN_DIR/f"fold{fold}_{arm}_history.csv"

    if cache.exists():
        print("✅ reuse", cache.name)
        with np.load(cache) as z:
            return {k:z[k] for k in z.files}

    m=inner_maps[fold]

    tr=train[train.group_key.isin(m["train_groups"])].copy().reset_index(drop=True)
    ca=train[train.group_key.isin(m["calib_groups"])].copy().reset_index(drop=True)
    ou=train[train.group_key.isin(m["outer_groups"])].copy().reset_index(drop=True)

    arm_seed = SEED + fold*1000
    # Same initialization seed for control and treatment.
    model = build_model(arm_seed).to("cuda")
    opt = build_optimizer(model)

    steps_per_epoch = math.ceil(len(tr)/TRAIN_BATCH)
    total_opt_steps = math.ceil(steps_per_epoch/ACCUM)*EPOCHS

    sched = get_linear_schedule_with_warmup(
        opt,
        int(total_opt_steps*WARMUP_RATIO),
        total_opt_steps
    )

    scaler = torch.amp.GradScaler("cuda")

    ca_loader=make_eval_loader(ca, True)
    ou_loader=make_eval_loader(ou, True)
    v500_loader=make_eval_loader(valid500, True)
    test_loader=make_eval_loader(test, False)

    hist=[]
    snap_ca=[]
    snap_ou=[]
    snap_v500=[]
    snap_test=[]

    arm_start=time.time()

    for epoch in range(1,EPOCHS+1):
        # Matched deterministic epoch seed and matched sample order across arms.
        epoch_seed = arm_seed + epoch*17
        seed_all(epoch_seed)

        sampler = WeightedRandomSampler(
            torch.tensor(sample_weights(tr), dtype=torch.double),
            num_samples=len(tr),
            replacement=True,
            generator=torch.Generator().manual_seed(epoch_seed),
        )

        tr_loader = DataLoader(
            MultiLabelDS(tr, True),
            batch_size=TRAIN_BATCH,
            sampler=sampler,
            num_workers=0,
            pin_memory=True,
            collate_fn=collator,
        )

        model.train()
        opt.zero_grad(set_to_none=True)

        running=0.0
        bce_running=0.0
        kl_running=0.0
        nrows=0

        for step,batch in enumerate(
            tqdm(tr_loader, desc=f"fold {fold} {arm} e{epoch}/{EPOCHS}")
        ):
            y=batch.pop("labels").to("cuda",non_blocking=True)
            inp={k:v.to("cuda",non_blocking=True) for k,v in batch.items()}

            with torch.autocast(device_type="cuda",dtype=torch.float16):
                logits1=model(**inp).logits
                logits2=model(**inp).logits

                bce=0.5*(
                    weighted_multilabel_bce(logits1,y)
                    +weighted_multilabel_bce(logits2,y)
                )

                if arm=="rdrop":
                    kl=symmetric_bernoulli_kl(logits1,logits2)
                    loss=bce + RDROP_ALPHA*kl
                else:
                    kl=logits1.new_tensor(0.0)
                    loss=bce

            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss at fold={fold} arm={arm}")

            scaler.scale(loss/ACCUM).backward()

            running += float(loss.item())*len(y)
            bce_running += float(bce.item())*len(y)
            kl_running += float(kl.item())*len(y)
            nrows += len(y)

            if (step+1)%ACCUM==0 or (step+1)==len(tr_loader):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

                old_scale=scaler.get_scale()
                scaler.step(opt)
                scaler.update()

                if scaler.get_scale() >= old_scale:
                    sched.step()

                opt.zero_grad(set_to_none=True)

        rec={
            "fold":fold,
            "arm":arm,
            "epoch":epoch,
            "loss":running/max(nrows,1),
            "bce":bce_running/max(nrows,1),
            "kl":kl_running/max(nrows,1),
            "elapsed_min":(time.time()-arm_start)/60.0,
        }
        hist.append(rec)
        pd.DataFrame(hist).to_csv(hist_path,index=False)
        print(rec)

        if epoch in SNAPSHOT_EPOCHS:
            snap_ca.append(predict(model,ca_loader))
            snap_ou.append(predict(model,ou_loader))
            snap_v500.append(predict(model,v500_loader))
            snap_test.append(predict(model,test_loader))

    out={
        "calib_ids":ca.ID.astype(str).to_numpy(),
        "outer_ids":ou.ID.astype(str).to_numpy(),
        "calib_y":ca[LABELS].to_numpy(np.int8),
        "outer_y":ou[LABELS].to_numpy(np.int8),
        "calib_prob":np.mean(snap_ca,axis=0).astype(np.float32),
        "outer_prob":np.mean(snap_ou,axis=0).astype(np.float32),
        "valid500_prob":np.mean(snap_v500,axis=0).astype(np.float32),
        "test_prob":np.mean(snap_test,axis=0).astype(np.float32),
        "arm_elapsed_min":np.array([(time.time()-arm_start)/60.0],dtype=np.float32),
    }

    np.savez_compressed(cache, **out)

    del model,opt,sched,scaler,ca_loader,ou_loader,v500_loader,test_loader
    gc.collect()
    torch.cuda.empty_cache()

    return out


In [ ]:

# 12. Execute all six matched fits and evaluate each outer fold ONCE

experiment_start=time.time()

fold_results=[]
raw={}

for fold in range(N_OUTER):
    raw[fold]={}

    # Fixed arm order is pre-registered.
    for arm in ["control","rdrop"]:
        raw[fold][arm]=train_arm(fold,arm)

    ctrl=raw[fold]["control"]
    trt=raw[fold]["rdrop"]

    assert ctrl["outer_ids"].tolist()==trt["outer_ids"].tolist()
    assert np.array_equal(ctrl["outer_y"],trt["outer_y"])
    assert ctrl["calib_ids"].tolist()==trt["calib_ids"].tolist()

    y_ca=ctrl["calib_y"].astype(int)
    y_ou=ctrl["outer_y"].astype(int)

    th_c=calibrate_thresholds(y_ca,ctrl["calib_prob"])
    th_t=calibrate_thresholds(y_ca,trt["calib_prob"])

    pred_c=apply_thresholds(ctrl["outer_prob"],th_c)
    pred_t=apply_thresholds(trt["outer_prob"],th_t)

    full_c=official_score(y_ou,pred_c)
    full_t=official_score(y_ou,pred_t)

    # Primary isolated comparison:
    # treatment toxicity + CONTROL attributes.
    toxonly=pred_c.copy()
    toxonly[:,0]=pred_t[:,0]
    toxonly_score=official_score(y_ou,toxonly)

    tox_c=toxic_macro_f1(y_ou[:,0],pred_c[:,0])
    tox_t=toxic_macro_f1(y_ou[:,0],pred_t[:,0])

    ap_c=average_precision_score(y_ou[:,0],ctrl["outer_prob"][:,0])
    ap_t=average_precision_score(y_ou[:,0],trt["outer_prob"][:,0])

    # script-specific toxicity
    outer_ids=ctrl["outer_ids"].astype(str)
    od=train.set_index("ID").loc[outer_ids].reset_index()

    script_stats={}
    for script_name in ["native","mixed","no_bengali"]:
        mask=(od.script.to_numpy()==script_name)
        if mask.sum():
            script_stats[f"{script_name}_control_tox_macro_f1"]=toxic_macro_f1(
                y_ou[mask,0],pred_c[mask,0]
            )
            script_stats[f"{script_name}_rdrop_tox_macro_f1"]=toxic_macro_f1(
                y_ou[mask,0],pred_t[mask,0]
            )
            script_stats[f"{script_name}_rows"]=int(mask.sum())

    rec={
        "fold":fold,
        "control_official":full_c,
        "rdrop_full_official":full_t,
        "toxonly_official":toxonly_score,
        "primary_gain":toxonly_score-full_c,
        "control_toxic_macro_f1":tox_c,
        "rdrop_toxic_macro_f1":tox_t,
        "toxic_macro_f1_gain":tox_t-tox_c,
        "control_toxic_ap":ap_c,
        "rdrop_toxic_ap":ap_t,
        "toxic_ap_gain":ap_t-ap_c,
        "control_threshold_toxic":th_c[0],
        "rdrop_threshold_toxic":th_t[0],
        **script_stats,
    }

    fold_results.append(rec)

fold_report=pd.DataFrame(fold_results)
display(fold_report.round(6))
fold_report.to_csv(EXPORT_DIR/"v48_fold_results.csv",index=False)

print("Elapsed experiment minutes:",(time.time()-experiment_start)/60.0)


In [ ]:

# 13. Build pooled OOF predictions and full diagnostic tables

# Allocate row-aligned OOF arrays.
ctrl_oof=np.full((len(train),7),np.nan,np.float32)
rdrop_oof=np.full((len(train),7),np.nan,np.float32)
ctrl_pred_oof=np.full((len(train),7),-1,np.int8)
rdrop_pred_oof=np.full((len(train),7),-1,np.int8)
fold_col=np.full(len(train),-1,np.int8)

fold_thresholds={"control":[],"rdrop":[]}

id_to_idx={id_:i for i,id_ in enumerate(train.ID.astype(str))}

for fold in range(N_OUTER):
    ctrl=raw[fold]["control"]
    trt=raw[fold]["rdrop"]

    y_ca=ctrl["calib_y"].astype(int)
    th_c=calibrate_thresholds(y_ca,ctrl["calib_prob"])
    th_t=calibrate_thresholds(y_ca,trt["calib_prob"])

    fold_thresholds["control"].append(th_c)
    fold_thresholds["rdrop"].append(th_t)

    ids=ctrl["outer_ids"].astype(str)
    idx=np.array([id_to_idx[x] for x in ids],int)

    ctrl_oof[idx]=ctrl["outer_prob"]
    rdrop_oof[idx]=trt["outer_prob"]

    ctrl_pred_oof[idx]=apply_thresholds(ctrl["outer_prob"],th_c)
    rdrop_pred_oof[idx]=apply_thresholds(trt["outer_prob"],th_t)
    fold_col[idx]=fold

assert np.isfinite(ctrl_oof).all()
assert np.isfinite(rdrop_oof).all()
assert (ctrl_pred_oof>=0).all()
assert (rdrop_pred_oof>=0).all()
assert (fold_col>=0).all()

y_all=train[LABELS].to_numpy(int)

ctrl_metric=per_label_metrics(y_all,ctrl_pred_oof,ctrl_oof)
rdrop_metric=per_label_metrics(y_all,rdrop_pred_oof,rdrop_oof)

pooled=ctrl_metric.merge(
    rdrop_metric,
    on="label",
    suffixes=("_control","_rdrop")
)

pooled["f1_gain"]=pooled.f1_metric_rdrop-pooled.f1_metric_control
pooled["ap_gain"]=pooled.ap_rdrop-pooled.ap_control

display(pooled.round(6))
pooled.to_csv(EXPORT_DIR/"v48_pooled_label_metrics.csv",index=False)

oof_export=pd.DataFrame({
    "ID":train.ID.astype(str),
    "outer_fold":fold_col,
    "group_key":train.group_key,
    "script":train.script,
})

for j,l in enumerate(LABELS):
    oof_export[f"{l}_y"]=y_all[:,j]
    oof_export[f"{l}_control_prob"]=ctrl_oof[:,j]
    oof_export[f"{l}_rdrop_prob"]=rdrop_oof[:,j]
    oof_export[f"{l}_control_pred"]=ctrl_pred_oof[:,j]
    oof_export[f"{l}_rdrop_pred"]=rdrop_pred_oof[:,j]

oof_export.to_csv(EXPORT_DIR/"v48_oof_predictions.csv",index=False)


In [ ]:

# 14. Paired GROUP bootstrap for the isolated toxicity contribution

rng=np.random.default_rng(SEED+777)

group_to_indices={}
for g,idxs in train.groupby("group_key").groups.items():
    group_to_indices[g]=np.asarray(list(idxs),int)

unique_groups=np.array(list(group_to_indices.keys()),dtype=object)
boot_diffs=[]

for b in range(1000):
    sampled=rng.choice(
        unique_groups,
        size=len(unique_groups),
        replace=True
    )

    idx=np.concatenate([
        group_to_indices[g]
        for g in sampled
    ])

    yc=y_all[idx]
    pc=ctrl_pred_oof[idx]
    pt=rdrop_pred_oof[idx]

    toxonly=pc.copy()
    toxonly[:,0]=pt[:,0]

    boot_diffs.append(
        official_score(yc,toxonly)-official_score(yc,pc)
    )

boot_diffs=np.asarray(boot_diffs,float)

bootstrap_summary={
    "mean":float(boot_diffs.mean()),
    "p2_5":float(np.quantile(boot_diffs,.025)),
    "p50":float(np.quantile(boot_diffs,.50)),
    "p97_5":float(np.quantile(boot_diffs,.975)),
    "prob_positive":float((boot_diffs>0).mean()),
}

print(json.dumps(bootstrap_summary,indent=2))


In [ ]:

# 15. Secondary diagnostic on the historical 500-row validation set
#
# No V48 model-selection decision is made from these labels.
# We ensemble the 3 fold models and use the MEDIAN inner-calibration threshold
# across folds, both pre-specified.

ctrl_v500=np.mean(
    [raw[f]["control"]["valid500_prob"] for f in range(N_OUTER)],
    axis=0
)
rdrop_v500=np.mean(
    [raw[f]["rdrop"]["valid500_prob"] for f in range(N_OUTER)],
    axis=0
)

ctrl_test=np.mean(
    [raw[f]["control"]["test_prob"] for f in range(N_OUTER)],
    axis=0
)
rdrop_test=np.mean(
    [raw[f]["rdrop"]["test_prob"] for f in range(N_OUTER)],
    axis=0
)

median_th_c=np.median(np.stack(fold_thresholds["control"]),axis=0)
median_th_t=np.median(np.stack(fold_thresholds["rdrop"]),axis=0)

y500=valid500[LABELS].to_numpy(int)
pred500_c=apply_thresholds(ctrl_v500,median_th_c)
pred500_t=apply_thresholds(rdrop_v500,median_th_t)

toxonly500=pred500_c.copy()
toxonly500[:,0]=pred500_t[:,0]

secondary={
    "control_official":official_score(y500,pred500_c),
    "rdrop_full_official":official_score(y500,pred500_t),
    "toxonly_official":official_score(y500,toxonly500),
    "primary_gain":official_score(y500,toxonly500)-official_score(y500,pred500_c),
    "control_toxic_macro_f1":toxic_macro_f1(y500[:,0],pred500_c[:,0]),
    "rdrop_toxic_macro_f1":toxic_macro_f1(y500[:,0],pred500_t[:,0]),
    "control_toxic_ap":average_precision_score(y500[:,0],ctrl_v500[:,0]),
    "rdrop_toxic_ap":average_precision_score(y500[:,0],rdrop_v500[:,0]),
}

print("SECONDARY ONLY:")
print(json.dumps(secondary,indent=2))


In [ ]:

# 16. Apply the PRE-REGISTERED V48 acceptance / rejection gate

mean_primary=float(fold_report.primary_gain.mean())
mean_tox_gain=float(fold_report.toxic_macro_f1_gain.mean())
wins=int((fold_report.primary_gain>0).sum())
worst_primary=float(fold_report.primary_gain.min())
mean_ap_gain=float(fold_report.toxic_ap_gain.mean())
worst_ap_gain=float(fold_report.toxic_ap_gain.min())

# Pooled script subgroup stability
script_checks={}
for script_name in ["native","mixed","no_bengali"]:
    mask=(train.script.to_numpy()==script_name)
    c=toxic_macro_f1(y_all[mask,0],ctrl_pred_oof[mask,0])
    t=toxic_macro_f1(y_all[mask,0],rdrop_pred_oof[mask,0])
    script_checks[script_name]={
        "control":c,
        "rdrop":t,
        "gain":t-c,
        "rows":int(mask.sum()),
    }

implicit_gain=float(
    pooled.loc[pooled.label=="implicit_toxicity","f1_gain"].iloc[0]
)
threat_gain=float(
    pooled.loc[pooled.label=="threatening","f1_gain"].iloc[0]
)

elapsed_total_min=float((time.time()-experiment_start)/60.0)

criteria={
    "mean_primary_gain_ge_0.004": mean_primary >= 0.004,
    "mean_toxic_macro_f1_gain_ge_0.008": mean_tox_gain >= 0.008,
    "positive_primary_in_at_least_2_of_3": wins >= 2,
    "no_fold_primary_below_minus_0.005": worst_primary >= -0.005,
    "mean_toxic_ap_gain_nonnegative": mean_ap_gain >= 0.0,
    "no_fold_toxic_ap_gain_below_minus_0.01": worst_ap_gain >= -0.01,
    "native_script_not_below_minus_0.015": script_checks["native"]["gain"] >= -0.015,
    "mixed_script_not_below_minus_0.015": script_checks["mixed"]["gain"] >= -0.015,
    "no_bengali_script_not_below_minus_0.015": script_checks["no_bengali"]["gain"] >= -0.015,
    "implicit_f1_not_below_minus_0.02": implicit_gain >= -0.02,
    "threat_f1_not_below_minus_0.02": threat_gain >= -0.02,
    "runtime_le_300_min": elapsed_total_min <= 300.0,
    "zero_group_leakage": True,
}

PASS=bool(all(criteria.values()))

summary={
    "protected_public_lb":PROTECTED_PUBLIC_LB,
    "new_minimum_public_lb_goal":MIN_PUBLIC_LB_GOAL,
    "main_public_lb_goal":MAIN_PUBLIC_LB_GOAL,
    "model_repo":MODEL_REPO,
    "model_revision":MODEL_REVISION,
    "model_license_note":MODEL_LICENSE_NOTE,
    "mean_primary_gain":mean_primary,
    "mean_toxic_macro_f1_gain":mean_tox_gain,
    "fold_wins":wins,
    "worst_primary_gain":worst_primary,
    "mean_toxic_ap_gain":mean_ap_gain,
    "worst_toxic_ap_gain":worst_ap_gain,
    "script_checks":script_checks,
    "implicit_f1_gain_pooled":implicit_gain,
    "threat_f1_gain_pooled":threat_gain,
    "bootstrap":bootstrap_summary,
    "secondary_500_validation":secondary,
    "elapsed_total_min":elapsed_total_min,
    "criteria":criteria,
    "PASS":PASS,
    "decision":(
        "PASS: eligible for a separate conservative V38 toxicity-integration check; "
        "NOT automatic champion replacement."
        if PASS else
        "REJECT: retain V38. Do not create/submit a Kaggle challenger from V48."
    ),
}

print(json.dumps(summary,indent=2))

with open(EXPORT_DIR/"v48_summary.json","w") as f:
    json.dump(summary,f,indent=2)


In [ ]:

# 17. Save ID-bearing validation/test probabilities, hashes, environment and package results

prob_export_valid=pd.DataFrame({
    "ID":valid500.ID.astype(str)
})
prob_export_test=pd.DataFrame({
    "ID":test.ID.astype(str)
})

for j,l in enumerate(LABELS):
    prob_export_valid[f"{l}_control_prob"]=ctrl_v500[:,j]
    prob_export_valid[f"{l}_rdrop_prob"]=rdrop_v500[:,j]
    prob_export_test[f"{l}_control_prob"]=ctrl_test[:,j]
    prob_export_test[f"{l}_rdrop_prob"]=rdrop_test[:,j]

prob_export_valid.to_csv(
    EXPORT_DIR/"v48_validation500_probabilities.csv",
    index=False
)
prob_export_test.to_csv(
    EXPORT_DIR/"v48_test_probabilities.csv",
    index=False
)

# Preserve protected champion exactly.
champ.to_csv(
    EXPORT_DIR/"submission_v38_PROTECTED.csv",
    index=False
)

import transformers

environment={
    "python":platform.python_version(),
    "torch":torch.__version__,
    "transformers":transformers.__version__,
    "sklearn":sklearn.__version__,
    "pandas":pd.__version__,
    "numpy":np.__version__,
    "gpu":torch.cuda.get_device_name(0),
}

config={
    "seed":SEED,
    "labels":LABELS,
    "model_repo":MODEL_REPO,
    "model_revision":MODEL_REVISION,
    "model_license_note":MODEL_LICENSE_NOTE,
    "model_hashes":model_hashes,
    "n_outer":N_OUTER,
    "calib_fraction":CALIB_FRACTION,
    "max_length":MAX_LENGTH,
    "train_batch":TRAIN_BATCH,
    "eval_batch":EVAL_BATCH,
    "accum":ACCUM,
    "epochs":EPOCHS,
    "snapshot_epochs":SNAPSHOT_EPOCHS,
    "backbone_lr":BACKBONE_LR,
    "head_lr":HEAD_LR,
    "weight_decay":WEIGHT_DECAY,
    "warmup_ratio":WARMUP_RATIO,
    "rdrop_alpha":RDROP_ALPHA,
    "threshold_grid":THRESH_GRID.tolist(),
    "environment":environment,
    "data_manifest":supplied_manifest,
}

with open(EXPORT_DIR/"v48_config.json","w") as f:
    json.dump(config,f,indent=2)

# Human-readable decision note
decision_text = f'''
V48 matched R-Drop experiment

Protected champion: V38 = {PROTECTED_PUBLIC_LB}
New minimum LB goal: {MIN_PUBLIC_LB_GOAL}
Main LB goal: {MAIN_PUBLIC_LB_GOAL}

PASS = {PASS}

{summary["decision"]}

Important:
- This notebook did NOT generate a Kaggle challenger submission.
- The 500-row validation set was secondary only.
- Test probabilities were saved only to avoid retraining if the experiment passes.
- If PASS=false, keep V38.
- If PASS=true, upload this ZIP to ChatGPT for a separate integration decision.
'''.strip()

(EXPORT_DIR/"README_DECISION.txt").write_text(decision_text)

zip_path=Path("/content/v48_results.zip")
with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in EXPORT_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print(decision_text)
print()
print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
